# SPARCS PR2 – Preparació de dades (versió corregida)

Aquest notebook està adaptat als **noms reals de les columnes** del teu fitxer SPARCS 2024 i evita l'error de `KeyError` amb diagnòstics.

Genera aquests fitxers per a Flourish:
- `overview_kpis.csv`
- `cost_by_age_gender.csv`
- `cost_by_admission.csv`
- `severity_vs_cost.csv`
- `diagnosis_or_area.csv`


In [1]:

import pandas as pd
import numpy as np

# Canvia aquest nom si el teu fitxer es diu diferent
file_path = "SPARCS_2024.csv"

x = pd.read_csv(file_path, low_memory=False)
print("Dimensions:", x.shape)
print("\nPrimeres columnes:")
print(x.columns.tolist())


Dimensions: (2196737, 33)

Primeres columnes:
['Health Service Area', 'Hospital County', 'Operating Certificate Number', 'Permanent Facility Id', 'Facility Name', 'Age Group', 'Zip Code', 'Gender', 'Race', 'Ethnicity', 'Length of Stay', 'Type of Admission', 'Patient Disposition', 'Discharge Year', 'CCSR Diagnosis Code', 'CCSR Diagnosis Description', 'CCSR Procedure Code', 'CCSR Procedure Description', 'APR DRG Code', 'APR DRG Description', 'APR MDC Code', 'APR MDC Description', 'APR Severity of Illness Code', 'APR Severity of Illness Description', 'APR Risk of Mortality', 'APR Medical Surgical Description', 'Payment Typology 1', 'Payment Typology 2', 'Payment Typology 3', 'Birth Weight', 'Emergency Department Indicator', 'Total Charges', 'Total Costs']


## 1. Selecció de columnes i neteja bàsica

In [2]:

# Columnes reals confirmades al teu fitxer
needed_cols = [
    'Health Service Area',
    'Hospital County',
    'Facility Name',
    'Age Group',
    'Gender',
    'Length of Stay',
    'Type of Admission',
    'CCSR Diagnosis Description',
    'APR DRG Description',
    'APR Severity of Illness Description',
    'APR Risk of Mortality',
    'Total Charges',
    'Total Costs'
]

# Quedar-nos només amb les que existeixen
existing_cols = [c for c in needed_cols if c in x.columns]
x = x[existing_cols].copy()

# Conversió numèrica
x["Length of Stay"] = pd.to_numeric(x["Length of Stay"], errors="coerce")
x["Total Charges"] = pd.to_numeric(x["Total Charges"], errors="coerce")
x["Total Costs"] = pd.to_numeric(x["Total Costs"], errors="coerce")

# Neteja de categories
for c in ["Age Group","Gender","Type of Admission","APR Severity of Illness Description","APR Risk of Mortality","CCSR Diagnosis Description","APR DRG Description","Health Service Area","Hospital County","Facility Name"]:
    if c in x.columns:
        x[c] = x[c].fillna("Unknown").astype(str).str.strip()

# Filtre bàsic de qualitat
x = x[
    x["Total Costs"].notna() &
    x["Length of Stay"].notna() &
    (x["Length of Stay"] >= 0)
].copy()

# Mètrica derivada
x["cost_per_day"] = np.where(
    x["Length of Stay"] > 0,
    x["Total Costs"] / x["Length of Stay"],
    np.nan
)

print("Dimensions després de neteja:", x.shape)
x.head()


Dimensions després de neteja: (2194333, 14)


,Health Service Area,Hospital County,Facility Name,Age Group,Gender,Length of Stay,Type of Admission,CCSR Diagnosis Description,APR DRG Description,APR Severity of Illness Description,APR Risk of Mortality,Total Charges,Total Costs,cost_per_day
0,Hudson Valley,Westchester,WESTCHESTER MEDICAL CENTER,0-17,F,1.0,Emergency,FEVER,FEVER AND INFLAMMATORY CONDITIONS,Moderate,Minor,46814.00,6772.07,6772.070
1,New York City,Queens,FLUSHING HOSPITAL MEDICAL CENTER,0-17,M,2.0,Emergency,FEVER,FEVER AND INFLAMMATORY CONDITIONS,Moderate,Moderate,13490.00,15464.30,7732.150
2,New York City,New York,NEW YORK-PRESBYTERIAN HOSPITAL - NEW YORK WEIL...,70 or Older,M,2.0,Emergency,FEVER,FEVER AND INFLAMMATORY CONDITIONS,Moderate,Moderate,49503.16,9324.77,4662.385
3,New York City,New York,NEW YORK-PRESBYTERIAN HOSPITAL - COLUMBIA PRES...,0-17,F,1.0,Emergency,FEVER,FEVER AND INFLAMMATORY CONDITIONS,Minor,Minor,27827.66,7304.27,7304.270
4,New York City,New York,MOUNT SINAI WEST,18-29,F,1.0,Emergency,FEVER,FEVER AND INFLAMMATORY CONDITIONS,Moderate,Minor,32798.29,7948.10,7948.100


## 2. KPI generals

In [3]:

overview_kpis = pd.DataFrame({
    "metric": [
        "Total hospitalitzacions",
        "Cost mitjà",
        "Estada mitjana",
        "Cost per dia"
    ],
    "value": [
        round(len(x), 2),
        round(x["Total Costs"].mean(), 2),
        round(x["Length of Stay"].mean(), 2),
        round(x["cost_per_day"].mean(), 2)
    ]
})

overview_kpis.to_csv("overview_kpis.csv", index=False)
overview_kpis


,metric,value
0,Total hospitalitzacions,2194333.00
1,Cost mitjà,25660.74
2,Estada mitjana,5.69
3,Cost per dia,5792.36


## 3. Cost per edat i gènere

In [4]:

cost_by_age_gender = (
    x.groupby(["Age Group", "Gender"], dropna=False)
     .agg(
         discharges=("Gender", "size"),
         avg_cost=("Total Costs", "mean"),
         avg_los=("Length of Stay", "mean"),
         cost_per_day=("cost_per_day", "mean")
     )
     .reset_index()
)

for col in ["avg_cost", "avg_los", "cost_per_day"]:
    cost_by_age_gender[col] = cost_by_age_gender[col].round(2)

cost_by_age_gender.to_csv("cost_by_age_gender.csv", index=False)
cost_by_age_gender.head(20)


,Age Group,Gender,discharges,avg_cost,avg_los,cost_per_day
0,0-17,F,145719,14491.32,3.83,3453.40
1,0-17,M,155647,15566.89,3.86,3728.61
2,0-17,U,129,7941.84,2.85,2636.18
3,18-29,F,133108,16258.97,3.78,5365.67
4,18-29,M,49868,26884.19,7.12,5548.30
5,18-29,U,58,18629.94,6.00,4742.83
6,30-49,F,268787,19156.31,4.20,5850.23
7,30-49,M,153909,27296.50,6.57,5674.73
8,30-49,U,104,17964.55,3.95,7387.83
9,50-69,F,262493,30094.05,6.23,6735.16


## 4. Cost per tipus d’ingrés

In [5]:

severity_order = {
    "Minor": 1,
    "Moderate": 2,
    "Major": 3,
    "Extreme": 4,
    "Unknown": np.nan
}

x["severity_num"] = x["APR Severity of Illness Description"].map(severity_order)

cost_by_admission = (
    x.groupby("Type of Admission", dropna=False)
     .agg(
         discharges=("Type of Admission", "size"),
         avg_cost=("Total Costs", "mean"),
         avg_los=("Length of Stay", "mean"),
         avg_cost_per_day=("cost_per_day", "mean"),
         avg_severity_score=("severity_num", "mean")
     )
     .reset_index()
)

for col in ["avg_cost", "avg_los", "avg_cost_per_day", "avg_severity_score"]:
    cost_by_admission[col] = cost_by_admission[col].round(2)

cost_by_admission.to_csv("cost_by_admission.csv", index=False)
cost_by_admission


,Type of Admission,discharges,avg_cost,avg_los,avg_cost_per_day,avg_severity_score
0,Elective,355519,30240.22,4.77,10774.00,1.80
1,Emergency,1458774,25955.19,6.10,4965.78,2.33
2,Newborn,197997,10227.62,3.42,2356.14,1.42
3,Not Available,3639,26005.23,8.02,4042.87,2.24
4,Trauma,10041,39523.66,7.06,6894.39,2.47
5,Urgent,168363,30754.64,6.58,6448.02,2.05


## 5. Severitat i cost

In [6]:

severity_vs_cost = (
    x.groupby(
        ["APR Severity of Illness Description", "APR Risk of Mortality"],
        dropna=False
    )
    .agg(
        discharges=("APR Severity of Illness Description", "size"),
        avg_cost=("Total Costs", "mean"),
        avg_los=("Length of Stay", "mean"),
        avg_cost_per_day=("cost_per_day", "mean")
    )
    .reset_index()
)

for col in ["avg_cost", "avg_los", "avg_cost_per_day"]:
    severity_vs_cost[col] = severity_vs_cost[col].round(2)

severity_vs_cost.to_csv("severity_vs_cost.csv", index=False)
severity_vs_cost.head(20)


,APR Severity of Illness Description,APR Risk of Mortality,discharges,avg_cost,avg_los,avg_cost_per_day
0,Extreme,Extreme,127389,83648.31,15.03,6123.30
1,Extreme,Major,48389,63798.10,13.58,5020.80
2,Extreme,Minor,1103,73276.05,20.97,5127.23
3,Extreme,Moderate,4917,62061.12,14.08,5952.06
4,Major,Extreme,41624,35942.62,7.72,5265.18
5,Major,Major,309396,33324.68,7.49,5156.07
6,Major,Minor,68125,24904.28,5.81,5096.76
7,Major,Moderate,153772,29887.38,6.53,5564.36
8,Minor,Extreme,20,11569.88,2.60,5064.68
9,Minor,Major,3388,19878.16,4.22,5861.14


## 6. Diagnòstics principals (corregit)

In [7]:

# Fem servir la columna correcta del teu fitxer
diagnosis_col = "CCSR Diagnosis Description"

# Comptar quins diagnòstics tenen més altes
diagnosis_counts = (
    x.groupby(diagnosis_col, dropna=False)
     .size()
     .reset_index(name="discharges")
     .sort_values("discharges", ascending=False)
)

# Excloure Unknown si existeix
diagnosis_counts = diagnosis_counts[diagnosis_counts[diagnosis_col] != "Unknown"]

# Top 12 diagnòstics més freqüents
top_diagnosis = diagnosis_counts.head(12)[diagnosis_col]

diagnosis_or_area = (
    x[x[diagnosis_col].isin(top_diagnosis)]
    .groupby(diagnosis_col, dropna=False)
    .agg(
        discharges=(diagnosis_col, "size"),
        avg_cost=("Total Costs", "mean"),
        avg_los=("Length of Stay", "mean"),
        avg_cost_per_day=("cost_per_day", "mean")
    )
    .reset_index()
    .rename(columns={diagnosis_col: "Diagnosis Group"})
    .sort_values("avg_cost", ascending=False)
)

for col in ["avg_cost", "avg_los", "avg_cost_per_day"]:
    diagnosis_or_area[col] = diagnosis_or_area[col].round(2)

diagnosis_or_area.to_csv("diagnosis_or_area.csv", index=False)
diagnosis_or_area.head(20)


,Diagnosis Group,discharges,avg_cost,avg_los,avg_cost_per_day
9,SEPTICEMIA,163329,39477.63,8.84,4572.51
10,SPONDYLOPATHIES/SPONDYLOARTHROPATHY (INCLUDING...,31907,37986.30,4.48,12757.17
8,SCHIZOPHRENIA SPECTRUM AND OTHER PSYCHOTIC DIS...,35629,35486.96,17.27,2139.90
2,CEREBRAL INFARCTION,31350,32666.94,6.84,5546.66
5,HEART FAILURE,62409,28856.37,6.69,4413.16
4,DIABETES MELLITUS WITH COMPLICATION,43779,28111.51,6.34,5215.69
1,CARDIAC DYSRHYTHMIAS,37074,25338.32,3.66,11531.56
7,PNEUMONIA (EXCEPT THAT CAUSED BY TUBERCULOSIS),37143,20540.53,5.21,4224.05
11,URINARY TRACT INFECTIONS,33958,16950.23,4.90,3989.22
0,ALCOHOL-RELATED DISORDERS,39827,13733.39,5.98,3060.03


## 7. Resum final

In [8]:

print("Fitxers generats correctament:")
for f in [
    "overview_kpis.csv",
    "cost_by_age_gender.csv",
    "cost_by_admission.csv",
    "severity_vs_cost.csv",
    "diagnosis_or_area.csv"
]:
    print("-", f)


Fitxers generats correctament:
- overview_kpis.csv
- cost_by_age_gender.csv
- cost_by_admission.csv
- severity_vs_cost.csv
- diagnosis_or_area.csv


In [9]:
import pandas as pd

age = pd.read_csv("cost_by_age_gender.csv")
age = age[age["Gender"].isin(["F", "M"])].copy()

age_order = ["0-17", "18-29", "30-49", "50-69", "70 or Older"]
age["Age Group"] = pd.Categorical(age["Age Group"], categories=age_order, ordered=True)
age = age.sort_values(["Age Group", "Gender"])

age_gender_metrics_long = age.melt(
    id_vars=["Age Group", "Gender", "discharges"],
    value_vars=["avg_cost", "cost_per_day", "avg_los"],
    var_name="metric",
    value_name="value"
)

age_gender_metrics_long.to_csv("age_gender_metrics_long.csv", index=False)
age_gender_metrics_long.head()

,Age Group,Gender,discharges,metric,value
0,0-17,F,145719,avg_cost,14491.32
1,0-17,M,155647,avg_cost,15566.89
2,18-29,F,133108,avg_cost,16258.97
3,18-29,M,49868,avg_cost,26884.19
4,30-49,F,268787,avg_cost,19156.31


In [10]:
adm = pd.read_csv("cost_by_admission.csv")
adm = adm[adm["Type of Admission"] != "Not Available"].copy()

adm = adm.rename(columns={
    "avg_cost": "cost",
    "avg_los": "los",
    "avg_cost_per_day": "cost_per_day",
    "avg_severity_score": "severity_score"
})

adm.to_csv("admission_tradeoff.csv", index=False)
adm.head()

,Type of Admission,discharges,cost,los,cost_per_day,severity_score
0,Elective,355519,30240.22,4.77,10774.00,1.80
1,Emergency,1458774,25955.19,6.10,4965.78,2.33
2,Newborn,197997,10227.62,3.42,2356.14,1.42
4,Trauma,10041,39523.66,7.06,6894.39,2.47
5,Urgent,168363,30754.64,6.58,6448.02,2.05


In [11]:
sev = pd.read_csv("severity_vs_cost.csv")

severity_order = ["Minor", "Moderate", "Major", "Extreme"]
risk_order = ["Minor", "Moderate", "Major", "Extreme"]

sev["APR Severity of Illness Description"] = pd.Categorical(
    sev["APR Severity of Illness Description"],
    categories=severity_order,
    ordered=True
)

sev["APR Risk of Mortality"] = pd.Categorical(
    sev["APR Risk of Mortality"],
    categories=risk_order,
    ordered=True
)

sev = sev.sort_values(
    ["APR Severity of Illness Description", "APR Risk of Mortality"]
)

sev.to_csv("severity_heatmap.csv", index=False)
sev.head()

,APR Severity of Illness Description,APR Risk of Mortality,discharges,avg_cost,avg_los,avg_cost_per_day
10,Minor,Minor,576888,12740.12,2.96,6155.40
11,Minor,Moderate,38847,17814.93,3.40,8206.40
9,Minor,Major,3388,19878.16,4.22,5861.14
8,Minor,Extreme,20,11569.88,2.60,5064.68
14,Moderate,Minor,461539,18387.23,4.82,5567.89


In [12]:
import pandas as pd

age = pd.read_csv("cost_by_age_gender_clean.csv")

age_gender_metrics_long = age.melt(
    id_vars=["Age Group", "Gender", "discharges"],
    value_vars=["avg_cost", "cost_per_day", "avg_los"],
    var_name="metric",
    value_name="value"
)

age_gender_metrics_long.to_csv("age_gender_metrics_long.csv", index=False)
age_gender_metrics_long.head()

,Age Group,Gender,discharges,metric,value
0,70 or Older,F,384561,avg_cost,27556.17
1,70 or Older,M,321878,avg_cost,30999.85
2,NaN,F,145719,avg_cost,14491.32
3,NaN,F,133108,avg_cost,16258.97
4,NaN,F,268787,avg_cost,19156.31


In [13]:
adm = pd.read_csv("cost_by_admission_clean.csv").copy()

adm = adm.rename(columns={
    "avg_cost": "cost",
    "avg_los": "los",
    "avg_cost_per_day": "cost_per_day",
    "avg_severity_score": "severity_score"
})

adm.to_csv("admission_tradeoff.csv", index=False)
adm.head()

,Type of Admission,discharges,cost,los,cost_per_day,severity_score
0,Trauma,10041,39523.66,7.06,6894.39,2.47
1,Urgent,168363,30754.64,6.58,6448.02,2.05
2,Elective,355519,30240.22,4.77,10774.00,1.80
3,Emergency,1458774,25955.19,6.10,4965.78,2.33
4,Newborn,197997,10227.62,3.42,2356.14,1.42


In [14]:
diag = pd.read_csv("diagnosis_or_area.csv").copy()

diag = diag.rename(columns={
    "avg_cost": "cost",
    "avg_los": "los",
    "avg_cost_per_day": "cost_per_day"
})

diag.to_csv("diagnosis_tradeoff.csv", index=False)
diag.head()

,Diagnosis Group,discharges,cost,los,cost_per_day
0,SEPTICEMIA,163329,39477.63,8.84,4572.51
1,SPONDYLOPATHIES/SPONDYLOARTHROPATHY (INCLUDING...,31907,37986.30,4.48,12757.17
2,SCHIZOPHRENIA SPECTRUM AND OTHER PSYCHOTIC DIS...,35629,35486.96,17.27,2139.90
3,CEREBRAL INFARCTION,31350,32666.94,6.84,5546.66
4,HEART FAILURE,62409,28856.37,6.69,4413.16


In [18]:
import pandas as pd

age_original = pd.read_csv("cost_by_age_gender.csv").copy()

age_clean = age_original[
    age_original["Gender"].isin(["F", "M"])
].copy()

age_order = ["0-17", "18-29", "30-49", "50-69", "70 or Older"]

age_clean["Age Group"] = pd.Categorical(
    age_clean["Age Group"],
    categories=age_order,
    ordered=True
)

age_clean = age_clean.sort_values(["Age Group", "Gender"])

age_clean.to_csv("cost_by_age_gender_clean_v2.csv", index=False)

print(age_clean["Age Group"].unique())
age_clean.head(20)

['0-17', '18-29', '30-49', '50-69', '70 or Older']
Categories (5, object): ['0-17' < '18-29' < '30-49' < '50-69' < '70 or Older']


,Age Group,Gender,discharges,avg_cost,avg_los,cost_per_day
0,0-17,F,145719,14491.32,3.83,3453.40
1,0-17,M,155647,15566.89,3.86,3728.61
3,18-29,F,133108,16258.97,3.78,5365.67
4,18-29,M,49868,26884.19,7.12,5548.30
6,30-49,F,268787,19156.31,4.20,5850.23
7,30-49,M,153909,27296.50,6.57,5674.73
9,50-69,F,262493,30094.05,6.23,6735.16
10,50-69,M,317978,32822.38,6.69,6678.30
12,70 or Older,F,384561,27556.17,6.31,5774.94
13,70 or Older,M,321878,30999.85,6.61,6448.91


In [19]:
age = pd.read_csv("cost_by_age_gender_clean_v2.csv").copy()

age_grid = age.melt(
    id_vars=["Age Group", "Gender", "discharges"],
    value_vars=["avg_cost", "cost_per_day", "avg_los"],
    var_name="metric",
    value_name="value"
)

age_grid.to_csv("age_grid_bar_chart_v2.csv", index=False)

print(age_grid["Age Group"].unique())
age_grid.head(20)

['0-17' '18-29' '30-49' '50-69' '70 or Older']


,Age Group,Gender,discharges,metric,value
0,0-17,F,145719,avg_cost,14491.32
1,0-17,M,155647,avg_cost,15566.89
2,18-29,F,133108,avg_cost,16258.97
3,18-29,M,49868,avg_cost,26884.19
4,30-49,F,268787,avg_cost,19156.31
5,30-49,M,153909,avg_cost,27296.50
6,50-69,F,262493,avg_cost,30094.05
7,50-69,M,317978,avg_cost,32822.38
8,70 or Older,F,384561,avg_cost,27556.17
9,70 or Older,M,321878,avg_cost,30999.85


In [20]:
import pandas as pd

adm = pd.read_csv("cost_by_admission_clean.csv").copy()

adm = adm.rename(columns={
    "avg_cost": "cost",
    "avg_los": "los",
    "avg_cost_per_day": "cost_per_day",
    "avg_severity_score": "severity_score"
})

adm.to_csv("admission_tradeoff.csv", index=False)
adm.head()

,Type of Admission,discharges,cost,los,cost_per_day,severity_score
0,Trauma,10041,39523.66,7.06,6894.39,2.47
1,Urgent,168363,30754.64,6.58,6448.02,2.05
2,Elective,355519,30240.22,4.77,10774.00,1.80
3,Emergency,1458774,25955.19,6.10,4965.78,2.33
4,Newborn,197997,10227.62,3.42,2356.14,1.42


In [21]:
import pandas as pd

diag = pd.read_csv("diagnosis_or_area.csv").copy()

diag = diag.rename(columns={
    "avg_cost": "cost",
    "avg_los": "los",
    "avg_cost_per_day": "cost_per_day"
})

diag.to_csv("diagnosis_tradeoff.csv", index=False)

diag.head(20)

,Diagnosis Group,discharges,cost,los,cost_per_day
0,SEPTICEMIA,163329,39477.63,8.84,4572.51
1,SPONDYLOPATHIES/SPONDYLOARTHROPATHY (INCLUDING...,31907,37986.30,4.48,12757.17
2,SCHIZOPHRENIA SPECTRUM AND OTHER PSYCHOTIC DIS...,35629,35486.96,17.27,2139.90
3,CEREBRAL INFARCTION,31350,32666.94,6.84,5546.66
4,HEART FAILURE,62409,28856.37,6.69,4413.16
5,DIABETES MELLITUS WITH COMPLICATION,43779,28111.51,6.34,5215.69
6,CARDIAC DYSRHYTHMIAS,37074,25338.32,3.66,11531.56
7,PNEUMONIA (EXCEPT THAT CAUSED BY TUBERCULOSIS),37143,20540.53,5.21,4224.05
8,URINARY TRACT INFECTIONS,33958,16950.23,4.90,3989.22
9,ALCOHOL-RELATED DISORDERS,39827,13733.39,5.98,3060.03


In [22]:
import pandas as pd

diag = pd.read_csv("diagnosis_or_area.csv").copy()

diag = diag.rename(columns={
    "avg_cost": "cost",
    "avg_los": "los",
    "avg_cost_per_day": "cost_per_day"
})

# Nova mètrica: càrrega econòmica aproximada
diag["economic_burden"] = diag["discharges"] * diag["cost"]

# Arrodonir si vols
diag["economic_burden"] = diag["economic_burden"].round(0)

diag.to_csv("diagnosis_quadrant.csv", index=False)

diag.head(20)

,Diagnosis Group,discharges,cost,los,cost_per_day,economic_burden
0,SEPTICEMIA,163329,39477.63,8.84,4572.51,6.447842e+09
1,SPONDYLOPATHIES/SPONDYLOARTHROPATHY (INCLUDING...,31907,37986.30,4.48,12757.17,1.212029e+09
2,SCHIZOPHRENIA SPECTRUM AND OTHER PSYCHOTIC DIS...,35629,35486.96,17.27,2139.90,1.264365e+09
3,CEREBRAL INFARCTION,31350,32666.94,6.84,5546.66,1.024109e+09
4,HEART FAILURE,62409,28856.37,6.69,4413.16,1.800897e+09
5,DIABETES MELLITUS WITH COMPLICATION,43779,28111.51,6.34,5215.69,1.230694e+09
6,CARDIAC DYSRHYTHMIAS,37074,25338.32,3.66,11531.56,9.393929e+08
7,PNEUMONIA (EXCEPT THAT CAUSED BY TUBERCULOSIS),37143,20540.53,5.21,4224.05,7.629369e+08
8,URINARY TRACT INFECTIONS,33958,16950.23,4.90,3989.22,5.755959e+08
9,ALCOHOL-RELATED DISORDERS,39827,13733.39,5.98,3060.03,5.469597e+08
